# **MScFE 690: CapstoneProject**
## **Influence Diagram as A Decision-Making Tool for Factor Investing**
### Student Group: **11186**
#### Members:
1. **Vahid Nikoofard**
2. **Dipanshu Sharma**
3. **Rhesa Prabowo Budhidarmo**


In [ ]:
# installing the required packages

In [2]:
# importing the required packages and libraries

import io, os, sys, math, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings 
warnings.filterwarnings('ignore')

In [4]:
# master project path
base_path = "/Users/sharmadipanshu/Developer/MScFE 2509 Capstone/WQU_Capstone_Project/MScFE_WQU_Capstone_Project"

# defining subpaths
inputs_raw = f"{base_path}/Inputs/raw"
inputs_clean = f"{base_path}/Inputs/clean"
outputs_eda = f"{base_path}/Outputs/eda"
meta_path = f"{base_path}/Inputs/meta"

In [5]:
# defining the required custom functions

## **Exploratory Data Analysis (EDA) / ETL (Extract, Transfom and Load)**

In [18]:
# loading the MSCI Factor excel dataset

msci_path = f"{inputs_raw}/MSCI_Emerging_Markets_Factor_Index.xlsx"
msci_factor_data = pd.read_excel(msci_path, header=5, index_col= 0, usecols="A:F", parse_dates=True)
print("The Period of the dataset is from {} to {}".format(msci_factor_data.index.min().date(), msci_factor_data.index.max().date()))
display(msci_factor_data.head())

# resampling the dataset to monthly frequency = last observation of each month
msci_monthly_data = msci_factor_data.resample("M").last()

The Period of the dataset is from 2000-05-31 to 2025-09-12


,MSCI EM (Emerging Markets) Momentum Index,MSCI EM (Emerging Markets) Growth Index,MSCI EM (Emerging Markets) Value Index,MSCI EM (Emerging Markets) Quality Index,MSCI EM (Emerging Markets) Minimum Volatility Index (USD)
Date,,,,,
2000-05-31,1605.533518,107.225547,87.816374,257.737087,275.493383
2000-06-01,1627.767814,108.493933,88.338474,258.757821,275.824320
2000-06-02,1677.599646,112.769957,90.180719,268.084459,281.948828
2000-06-05,1687.410963,113.305427,91.153167,268.287892,283.679510
2000-06-06,1683.573275,112.498659,90.604808,266.346813,282.923252


In [10]:
# handling the missing returns
n_obs = len(msci_monthly_data)
missing_frac = msci_monthly_data.isna().sum() / n_obs
missing_report = pd.DataFrame({"n_missing": msci_monthly_data.isna().sum(),"missing_frac": missing_frac}).sort_values("missing_frac", ascending=False)
display(missing_report)

,n_missing,missing_frac
MSCI EM (Emerging Markets) Momentum Index,1,0.003279
MSCI EM (Emerging Markets) Growth Index,1,0.003279
MSCI EM (Emerging Markets) Value Index,1,0.003279
MSCI EM (Emerging Markets) Quality Index,1,0.003279
MSCI EM (Emerging Markets) Minimum Volatility Index (USD),1,0.003279


In [20]:
# there is only one missing value in the dataset, using forward fill and backward fill method to impute it
msci_monthly_data = msci_monthly_data.ffill().bfill()

# calculating the monthly returns
msci_monthly_returns = msci_monthly_data.pct_change().dropna(how = 'any')
msci_monthly_returns.columns = [col + "_ret" for col in msci_monthly_returns.columns]
display(msci_monthly_returns.head())

# exported the cleaned dataset to the clean folder
msci_monthly_returns.to_csv(f"{inputs_clean}/msci_em_factor_monthly_returns.csv")

,MSCI EM (Emerging Markets) Momentum Index_ret,MSCI EM (Emerging Markets) Growth Index_ret,MSCI EM (Emerging Markets) Value Index_ret,MSCI EM (Emerging Markets) Quality Index_ret,MSCI EM (Emerging Markets) Minimum Volatility Index (USD)_ret
Date,,,,,
2000-06-30,0.010990,0.046680,0.020547,0.027184,0.023482
2000-07-31,-0.066766,-0.084035,-0.016382,-0.044815,-0.024922
2000-08-31,0.022835,0.013714,-0.005189,0.030152,-0.012023
2000-09-30,-0.101667,-0.088717,-0.085947,-0.076436,-0.049756
2000-10-31,-0.065633,-0.073891,-0.071061,-0.044771,-0.063982


In [21]:
# summary statistics + higher moments
msci_returns_summary_stats = msci_monthly_returns.describe().T
msci_returns_summary_stats['skew'] = msci_monthly_returns.skew()
msci_returns_summary_stats['kurtosis'] = msci_monthly_returns.kurtosis()
msci_returns_summary_stats['n_obs'] = msci_monthly_returns.count()
msci_returns_summary_stats['n_nan'] = msci_monthly_returns.isna().sum()

# saving the summary stats
msci_returns_summary_stats.to_csv(os.path.join(outputs_eda, "msci_returns_summary_stats.csv"))

# display
display(msci_returns_summary_stats)

,count,mean,std,min,25%,50%,75%,max,skew,kurtosis,n_obs,n_nan
MSCI EM (Emerging Markets) Momentum Index_ret,304.0,0.009541,0.062747,-0.248269,-0.024509,0.013225,0.051936,0.153438,-0.517298,1.042399,304,0
MSCI EM (Emerging Markets) Growth Index_ret,304.0,0.007416,0.061172,-0.276014,-0.025875,0.007555,0.047505,0.163168,-0.485446,1.382005,304,0
MSCI EM (Emerging Markets) Value Index_ret,304.0,0.008154,0.059276,-0.271224,-0.022856,0.008568,0.045140,0.179799,-0.356819,1.660964,304,0
MSCI EM (Emerging Markets) Quality Index_ret,304.0,0.008435,0.056072,-0.271266,-0.021210,0.009284,0.043682,0.174163,-0.529183,2.089155,304,0
MSCI EM (Emerging Markets) Minimum Volatility Index (USD)_ret,304.0,0.008358,0.045300,-0.221788,-0.016591,0.010955,0.036430,0.130229,-0.592636,2.127862,304,0
